In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import time

In [2]:
import pwd


%cd /Volumes/FastSSD_2/
%ls *.db

/Volumes/FastSSD_2
DEMO_2024.db               RX_2024.db
DX_2024.db                 ZIP3_all.db
ENROL_INTERVALS_2024.db    siblings_eligible_2023.db
PHE_2024.db                visibility_years2024.db
PX_2024.db


In [ ]:
# phe_con = sqlite3.connect('PHE_2024.db')
# c_phe = phe_con.cursor()
# script = """
# SELECT COUNT(enrolid) FROM PHE
# """

# all_patients_2024 = c_phe.execute(script).fetchone()[0]
# print(f"{all_patients_2024:,}")





In [3]:
vis_con = sqlite3.connect('visibility_years2024.db')
c_vis = vis_con.cursor()

# Get all table names
c_vis.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = c_vis.fetchall()

# Print table names
print("Tables in database:")
for table in tables:
    print(f"- {table[0]}")
    print(f"Columns in the table {table[0]}")
    c_vis.execute(f"PRAGMA table_info({table[0]});")
    columns = c_vis.fetchall()
    print(columns)


Tables in database:
- visibility_results
Columns in the table visibility_results
[(0, 'enrolid', 'INTEGER', 0, None, 0), (1, 'total_visibility', 'REAL', 0, None, 0), (2, 'visibility_2010', 'REAL', 0, None, 0)]


In [4]:

# Execute the SQL query

query = """
    SELECT enrolid 
    FROM visibility_results 
    WHERE total_visibility > 1
"""
c_vis.execute(query)

# Fetch the results and store them in a flat list
# cursor.fetchall() returns a list of tuples like: [(id1,), (id2,), (id3,)]
# We use a list comprehension to extract just the first element of each tuple
enrolids_more_than_1_year = [row[0] for row in c_vis.fetchall()] 
uniq_enrolids_more_than_1_year = set(enrolids_more_than_1_year) #unique patients only
print(f"Found {len(uniq_enrolids_more_than_1_year):,} patients.")
print(enrolids_more_than_1_year[:5]) # Print the first 5 to verify

Found 138,879,229 patients.
[39103, 62604, 105803, 132604, 194404]


In [14]:
enrol_con = sqlite3.connect('ENROL_INTERVALS_2024.db')
c_enrol = enrol_con.cursor()

# Get all table names
c_enrol.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = c_enrol.fetchall()

# Print table names
print("Tables in database:")
for table in tables:
    print(f"- {table[0]}")
    print(f"Columns in the table {table[0]}")
    c_enrol.execute(f"PRAGMA table_info({table[0]});")
    columns = c_enrol.fetchall()
    print(columns)


Tables in database:
- ENROL_INTERVAL
Columns in the table ENROL_INTERVAL
[(0, 'enrolid', 'INTEGER', 0, None, 0), (1, 'startday_int', 'INTEGER', 0, None, 0), (2, 'endday_int', 'INTEGER', 0, None, 0), (3, 'startyear', 'INTEGER', 0, None, 0), (4, 'startmonth', 'INTEGER', 0, None, 0), (5, 'startday', 'INTEGER', 0, None, 0), (6, 'endyear', 'INTEGER', 0, None, 0), (7, 'endmonth', 'INTEGER', 0, None, 0), (8, 'endday', 'INTEGER', 0, None, 0), (9, 'days_in_interval', 'INTEGER', 0, None, 0)]


In [15]:
demo_con = sqlite3.connect('DEMO_2024.db')
c_demo = demo_con.cursor()

# Get all table names
c_demo.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = c_demo.fetchall()

# Print table names
print("Tables in database:")
for table in tables:
    print(f"- {table[0]}")
    print(f"Columns in the table {table[0]}")
    c_demo.execute(f"PRAGMA table_info({table[0]});")
    columns = c_demo.fetchall()
    print(columns)


Tables in database:
- DEMO
Columns in the table DEMO
[(0, 'enrolid', 'INTEGER', 0, None, 0), (1, 'efamid', 'INTEGER', 0, None, 0), (2, 'dobyr', 'INTEGER', 0, None, 0), (3, 'sex', 'INTEGER', 0, None, 0)]


In [16]:
# # Attach your DEMOGRAPHICS database (the one with Date of Birth)
# # We give it an alias 'db2' so we can reference it in the query
# c_enrol.execute("ATTACH DATABASE 'DEMO_2024.db' AS db2")

# # Write the query to JOIN the two tables across the databases

# query = """
#     SELECT e.enrolid 
#     FROM main.ENROL_INTERVAL e
#     JOIN db2.DEMO d ON e.enrolid = d.enrolid
#     WHERE (e.startyear - d.dobyr) >= 20
# """


# # Execute and fetch into a list
# c_enrol.execute(query)
# more_than_20_at_enrolment = [row[0] for row in c_enrol.fetchall()]


# print(f"Found {len(more_than_20_at_enrolment)} patients who were >= 20 at enrollment.")

In [17]:
%pwd

'/Volumes/FastSSD_2'

In [18]:
# the script above was implemented via get_adult_patients.py
# here is the output

# Load the file back into a variable
with open('/Users/annagerasimenko/more_than_20_at_enrolment.pkl', 'rb') as f:
    more_than_20_at_enrolment = pickle.load(f)

# Verify the data
print(f"Loaded {len(more_than_20_at_enrolment):,} IDs.")

Loaded 179,415,164 IDs.


In [19]:
# combining those who are more than 20 at enrolment and those who have visibility > 1 year
adult_patients = set(more_than_20_at_enrolment) & set(enrolids_more_than_1_year)
print(f"Found {len(adult_patients):,} patients who are both >= 20 at enrollment and have visibility > 1 year.")



Found 100,133,299 patients who are both >= 20 at enrollment and have visibility > 1 year.


In [8]:
#now I need to get patients who have diabetes
phe_con = sqlite3.connect('PHE_2024.db')
c_phe = phe_con.cursor()
# Get all table names
c_phe.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = c_phe.fetchall()

# Print table names
print("Tables in database:")
for table in tables:
    print(f"- {table[0]}")
    print(f"Columns in the table {table[0]}")
    c_phe.execute(f"PRAGMA table_info({table[0]});")
    columns = c_phe.fetchall()
    print(columns)

# #getting all patients with diabetes

diab_phe = {"EM_202", "EM_202.1", "BI_181", "EM_202.2"} #phecode_X encoding

# Create a string with the correct number of question marks: "?, ?, ?, ..."
placeholders = ', '.join(['?'] * len(diab_phe))

# The SQL query just uses the placeholders
script = f"""
SELECT DISTINCT enrolid
FROM PHE
WHERE phecode IN ({placeholders})
"""

start_time = time.time()
# Convert the set to a tuple and pass it as the second argument to execute()
diab_patients = c_phe.execute(script, tuple(diab_phe)).fetchall()
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Time taken to execute the query: {elapsed_time:.2f} seconds")

Tables in database:
- PHE
Columns in the table PHE
[(0, 'enrolid', 'INTEGER', 0, None, 0), (1, 'intday', 'INTEGER', 0, None, 0), (2, 'phecode', 'TEXT', 0, None, 0)]
Time taken to execute the query: 78.94 seconds


In [10]:
print(f"{len(diab_patients):,}")
diab_patients = [row[0] for row in diab_patients]
print(diab_patients[:5])


18,110,999
[502, 24102, 82002, 85001, 85103]


In [ ]:
eligible_patients = set(adult_patients) & set(diab_patients)
with open('eligible_patients.pkl', 'wb') as f:
    pickle.dump(eligible_patients, f)




In [23]:
print(f"{len(eligible_patients):,}")

12,174,785


In [24]:
%pwd

'/Volumes/FastSSD_2'

In [5]:
with open('/Users/annagerasimenko/patient_info_phe_intday_2024.pkl', 'rb') as f:
    data = pickle.load(f)   
    phe_patients = data['phe_patients']
    all_dxs = data['all_dxs']


In [7]:
print(type(phe_patients))
print(type(all_dxs))

<class 'dict'>
<class 'set'>


In [9]:
print(list(all_dxs)[0:10])

['GE_965', 'NS_351.2', 'GI_509.4', 'CM_758.3', 'CM_776', 'MS_740', 'CV_443.2', 'GI_513', 'SS_840.12', 'GU_593.1']


In [ ]:
# See how many entries (patients) are in the dictionary
print(f"Number of keys: {len(phe_patients):,}")

# See the first 5 keys
# print("First 5 keys:", list(phe_patients.keys())[:5])

# See the value for the first key (to understand the structure)
first_key = list(phe_patients.keys())[0]
print(f"Key: {first_key}")
print(f"Value: {phe_patients[first_key]}")
print(f"Value type: {type(phe_patients[first_key])}")


Number of keys: 12,174,785
First 5 keys: [2902, 3702, 3902, 4601, 6002]
Key: 2902
Value: {'CV_401.1': [125, 140, 314, 329, 497, 514, 679, 496, 504, 518, 678, 1074, 2361, 2375, 2416, 3110, 3102, 3549, 3500, 3884, 4178, 4194, 4539, 4977, 4913, 5760, 6000, 6016, 6040, 6084, 6364, 6429, 6462], 'DE_670': [101, 623], 'ID_089.2': [101], 'EM_239': [329, 679, 6000, 6016, 6364, 6429, 6462], 'CA_138': [623], '?': [2361, 2212, 2533, 3198, 3515, 3591, 3884, 3893, 3962, 4301, 4539, 4543, 4977, 4913, 4976, 6000, 6016, 6054, 6076, 6020, 6364, 6429], 'GU_605': [2416, 3884, 4539], 'DE_660.122': [2508, 2536, 5760], 'DE_660.123': [2508], 'EM_204': [3110, 3549, 3521, 3893, 4542, 4602, 4923], 'CA_103': [3235], 'DE_672.21': [3235], 'MS_713.31': [3885, 3884, 3893, 3895, 3900, 3902, 3907, 3908], 'MS_727': [3888], 'MS_726.1': [3885, 3893, 4913], 'MS_708': [3885], 'DE_682.11': [4178], 'MS_721.11': [4178], 'CA_139.5': [4194], 'MS_718.5': [4539], 'CA_136.41': [4977, 4978], 'GI_523.2': [4977], 'GI_530.4': [4977], '